<img src="https://raw.githubusercontent.com/telmomm/CERTA/main/docs/assets/logo/certa-logo.png" alt="CERTA" width="220">

# Example notebook

*The notebook template applied to the identification counts of the systematic review*

Shows the template with real data: it reads the identification counts of T0.2 and summarises how many records each source contributes and how many are removed.

- **Input:** `../../../WP0-state-of-the-art/T0.2-search-and-deduplication/counts.csv`.
- **Output:** none; it only displays tables.
- **To re-run:** run all cells.
- **Environment:** the repository `.venv` (see `CONTRIBUTING.md`).

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 90)
CERTA = {"navy": "#0B2D4A", "teal": "#00BFA5", "blue": "#00AEEF", "sky": "#A7E3F7", "ice": "#E6EEF6"}

class table:
    """Shows a DataFrame with the CERTA colours in the notebook and as a LaTeX table in the PDF export.
    `bar` is a numeric column drawn as bars."""
    def __init__(self, df, bar=None):
        self.df, self.bar = df, bar
    def _repr_html_(self):
        styler = self.df.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": f"background-color: {CERTA['navy']}; color: white; text-align: left;"},
            {"selector": "tbody tr:nth-child(even)", "props": f"background-color: {CERTA['ice']};"},
        ])
        if self.bar is not None:
            styler = styler.bar(subset=[self.bar], color=CERTA["teal"], vmin=0)
        return styler.to_html()
    def _repr_latex_(self):
        return self.df.style.hide(axis="index").format(precision=3).to_latex(hrules=True)
    def __repr__(self):
        return self.df.to_string(index=False)

INPUT = Path("../../../WP0-state-of-the-art/T0.2-search-and-deduplication/counts.csv")

## 1. Load

In [2]:
counts = pd.read_csv(INPUT)
counts["item"] = counts["item"].str.strip()
table(counts)

item,records
Scopus,1380
Web of Science,801
IEEE Xplore X1,545
IEEE Xplore X2,488
Tracking: backward citations,0
Tracking: forward citations,692
Tracking: author,422
Total identified,4328
"Removed: no authors (front matter, proceedings, standards)",74
Removed: document type (tracking),7


## 2. Analysis

Records contributed by each source, as a share of the total identified.

In [3]:
total = counts.loc[counts["item"] == "Total identified", "records"].item()
sources = counts[counts["item"].str.match(r"Scopus|Web of Science|IEEE|Tracking")].copy()
sources["share"] = (sources["records"] / total).round(3)
table(sources, bar="records")

item,records,share
Scopus,1380,0.319000
Web of Science,801,0.185000
IEEE Xplore X1,545,0.126000
IEEE Xplore X2,488,0.113000
Tracking: backward citations,0,0.000000
Tracking: forward citations,692,0.160000
Tracking: author,422,0.098000


Records removed before screening, by reason.

In [4]:
removed = counts[counts["item"].str.startswith("Removed")].copy()
removed["reason"] = removed["item"].str.replace("Removed: ", "")
print(f"Removed: {removed['records'].sum()} of {total} ({removed['records'].sum() / total:.0%})")
table(removed[["reason", "records"]], bar="records")

Removed: 1696 of 4328 (39%)


reason,records
"no authors (front matter, proceedings, standards)",74
document type (tracking),7
duplicates,1615


## Conclusions and notes

- **Scopus is the largest source**, with 1380 of the 4328 records identified.
- **Duplicates are the main reason for removal**: 1615 records, well above the 74 without authors and the 7 removed by document type.

### Notes
